In [1]:
# 1. Import các thư viện cần thiết
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2. Khai báo label_list phù hợp với mô hình của bạn (cần sửa lại theo checkpoint bạn huấn luyện)
# Ví dụ cho NER tiếng Việt với PhoBERT, thường sẽ như sau:
label_list = [
    "B-AGE", "B-DATE", "B-GENDER", "B-JOB", "B-LOCATION", "B-NAME", "B-ORGANIZATION", "B-PATIENT_ID",
    "B-SYMPTOM_AND_DISEASE", "B-TRANSPORTATION", "I-AGE", "I-DATE", "I-JOB", "I-LOCATION", "I-NAME",
    "I-ORGANIZATION", "I-PATIENT_ID", "I-SYMPTOM_AND_DISEASE", "I-TRANSPORTATION", "O"
]

# Nếu bạn có label_list khác, hãy thay thế cho đúng!

# 3. Định nghĩa hàm split_sentences và predict_entities_for_text (đoạn code bạn gửi)
def split_sentences(text):
    # Tách câu theo dấu câu, giữ lại dấu câu ở cuối
    sentences = re.split(r'(?<=[.!?…])\s+', text.strip())
    # Loại bỏ câu rỗng
    sentences = [s for s in sentences if s.strip()]
    return sentences

# 4. Load model và tokenizer
model_dir = '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned'
model = AutoModelForTokenClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 5. Tạo id2label map
id2label = {i: label for i, label in enumerate(label_list)}


In [4]:
def merge_subwords(tokens, labels):
    words = []
    word_labels = []
    current_word = ""
    current_label = None
    for token, label in zip(tokens, labels):
        # Bỏ qua token đặc biệt
        if token in ["<s>", "</s>", "<pad>"]:
            continue
        # Token bắt đầu bằng ký tự đặc biệt (dành cho PhoBERT), ví dụ: Ġ hoặc space trước từ (nếu là Roberta/BART)
        if token.startswith("▁"):
            token = token[1:]
        if token.endswith("@@"):
            # BPE subword, bỏ @@ và nối vào current_word
            current_word += token[:-2]
            if current_label is None:
                current_label = label
        else:
            # Kết thúc 1 từ
            current_word += token
            if current_word:
                words.append(current_word)
                word_labels.append(current_label if current_label is not None else label)
            # Reset cho từ mới
            current_word = ""
            current_label = None
    return words, word_labels


In [6]:
def predict_entities_for_text(text):
    sentences = split_sentences(text)
    print(f"Phát hiện {len(sentences)} câu trong đoạn văn.")

    for idx, sent in enumerate(sentences, 1):
        print(f"\n------ Câu {idx} ------")
        inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)
        predicted_labels = [id2label[label.item()] for label in predictions[0]]
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        # Dùng hàm merge_subwords
        words, word_labels = merge_subwords(tokens, predicted_labels)
        print(f"{'Word':<20} {'Predicted Label'}")
        print("-" * 40)
        for word, label in zip(words, word_labels):
            print(f"{word:<20} {label}")

# 6. Nhập văn bản và thực hiện nhận diện thực thể
text = input("Nhập đoạn văn để nhận dạng thực thể: ")
predict_entities_for_text(text)



Nhập đoạn văn để nhận dạng thực thể: Bảy đang bị sốt
Phát hiện 1 câu trong đoạn văn.

------ Câu 1 ------
Word                 Predicted Label
----------------------------------------
Bảy                  B-NAME
đang                 O
bị                   O
sốt                  B-SYMPTOM_AND_DISEASE
